In [245]:
import os 
from dotenv import load_dotenv
from google import genai
from langchain.messages import SystemMessage, HumanMessage
from langchain.chat_models import init_chat_model

import json
from uuid import uuid4
from datetime import datetime


from pydantic import BaseModel
from typing import List
import random

import asyncio
from loguru import logger
from pathlib import Path


load_dotenv()

True

In [247]:
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

SCHEMA_VERSION = "1.0"
MODEL_NAME = "gemini-2.5-flash-lite"

OUTPUT_FILE = Path("dataset_output.jsonl")
OUTPUT_FILE.parent.mkdir(exist_ok=True, parents=True)

In [ ]:
model = init_chat_model(
    MODEL_NAME,
    model_provider="google-genai",
    temperature=0.7,
    max_tokens=1000,
    api_key = GEMINI_API_KEY
)

# shcemas

In [246]:
from enum import Enum
from typing import List
from pydantic import BaseModel, Field


# =========================================================
# ENUMS
# TTS-compatible values
# =========================================================

class Emotion(str, Enum):
    happy = "speak cheerfully and positively"
    sad = "speak sadly with a soft emotional tone"
    angry = "speak angrily with intensity"
    excited = "speak excitedly with enthusiasm"
    neutral = "speak naturally and calmly"
    confused = "speak with confusion and uncertainty"
    sarcastic = "speak sarcastically with a teasing tone"
    fearful = "speak with fear and urgency"


class SpeakerStyle(str, Enum):
    casual = "use a casual conversational tone"
    formal = "use a formal professional tone"
    customer_service = "speak like a polite customer service agent"
    phone_call = "speak like a natural phone conversation"
    street_talk = "use informal street-style dialogue"
    podcast = "speak like a podcast host"
    announcement = "speak like a public announcement"


class SpeakingRate(str, Enum):
    slow = "speaking_rate: slow"
    normal = "speaking_rate: normal"
    fast = "speaking_rate: fast"


class EnergyLevel(str, Enum):
    low = "with low energy"
    medium = "with moderate energy"
    high = "with high energy"


# =========================================================
# Background Noise
# Used later for audio augmentation / mixing
# =========================================================

class BackgroundNoise(str, Enum):

    none = "clean audio"

    street_noise = (
        "background street noise"
    )

  
    people_noise = (
        "background crowd"
    )

   
    wind_noise = (
        "light outdoor wind noise"
    )


class Category(str, Enum):
    daily_conversation = "daily_conversation"
    delivery = "delivery"
    shopping = "shopping"
    transportation = "transportation"
    customer_support = "customer_support"
    education = "education"
    healthcare = "healthcare"
    news = "news"
    restaurant = "restaurant"
    emergency = "emergency"


# =========================================================
# LLM OUTPUT SCHEMA
# =========================================================

class LLMGeneratedSchema(BaseModel):

    text: str = Field(
        ...,
        description=(
            "Natural Egyptian Arabic utterance suitable for STT training. "
            "Numbers must be written in words, not digits. "
            "English technical/product words should remain in English. "
            "Natural punctuation should be preserved."
        )
    )

    emotion: Emotion = Field(
        ...,
        description="Emotional speaking style for TTS generation."
    )

    speaker_style: SpeakerStyle = Field(
        ...,
        description="Speaker persona or conversation style."
    )

    speaking_rate: SpeakingRate = Field(
        ...,
        description="Speech speed."
    )

    energy: EnergyLevel = Field(
        ...,
        description="Voice energy and intensity."
    )

    background_noise: BackgroundNoise = Field(
        ...,
        description=(
            "Expected environmental background sound or recording condition."
        )
    )

    category: Category = Field(
        ...,
        description="Conversation domain or scenario."
    )

    code_switching: bool = Field(
        ...,
        description="Whether Arabic and English are mixed."
    )

    contains_numbers: bool = Field(
        ...,
        description="Whether numeric concepts are mentioned."
    )

    tags: List[str] = Field(
        ...,
        description="Extra semantic metadata tags."
    )

# Utils 

In [174]:
import re
import unicodedata


# =========================================================
# Arabic Text Normalization
# =========================================================

ARABIC_DIACRITICS = re.compile("""
    ّ    | # Shadda
    َ    | # Fatha
    ً    | # Tanwin Fath
    ُ    | # Damma
    ٌ    | # Tanwin Damm
    ِ    | # Kasra
    ٍ    | # Tanwin Kasr
    ْ    | # Sukun
    ـ      # Tatweel
""", re.VERBOSE)


# Emojis + symbols
EMOJI_PATTERN = re.compile(
    "["
    "\U0001F600-\U0001F64F"  # emoticons
    "\U0001F300-\U0001F5FF"  # symbols & pictographs
    "\U0001F680-\U0001F6FF"  # transport & map
    "\U0001F1E0-\U0001F1FF"  # flags
    "\U00002700-\U000027BF"
    "\U000024C2-\U0001F251"
    "]+",
    flags=re.UNICODE
)


# Remove weird/special characters
SPECIAL_CHARS_PATTERN = re.compile(
    r"[^؀-ۿa-zA-Z\s\.,!\؟،]"
)


DIGIT_PATTERN = re.compile(r"\d")


def normalize_text(text: str) -> str:
    """
    Normalize Egyptian Arabic text for STT datasets.

    - Remove emojis
    - Remove Arabic diacritics
    - Remove weird symbols
    - Remove tatweel
    - Normalize whitespace
    - Ensure no digits exist
    """

    # Unicode normalization
    text = unicodedata.normalize("NFKC", text)

    # Remove emojis
    text = EMOJI_PATTERN.sub("", text)

    # Remove tashkeel
    text = ARABIC_DIACRITICS.sub("", text)

    # Remove weird characters
    text = SPECIAL_CHARS_PATTERN.sub("", text)

    # Remove digits
    text = DIGIT_PATTERN.sub("", text)

    # Normalize spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text


def validate_no_digits(text: str) -> bool:
    """
    Ensure text contains no numeric digits.
    """
    return not bool(DIGIT_PATTERN.search(text))

In [227]:
def save_to_jsonl(record: dict):
    with open(OUTPUT_FILE, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False, default=str) + "\n")

# generation part

In [287]:
SYSTEM_PROMPT = """
Act like an expert Egyptian Arabic speech dataset designer and audio data annotation specialist.

Your goal is to generate highly realistic, production-quality utterances for Speech-To-Text (STT) training, strictly aligned with a predefined JSON schema.

CORE OBJECTIVE:
Produce a single, valid JSON object matching the LLMGeneratedSchema exactly, with no extra fields, no commentary, and no surrounding text.

HARD RULES (NON-NEGOTIABLE):
1. Output MUST be valid JSON only.
2. Use ONLY Egyptian Arabic dialect for the "text" field.
3. Text must sound natural, conversational, and spoken (not written Arabic).
4. Numbers MUST be written in words (never digits).
5. English technical/product terms MUST remain in English.
6. Avoid Modern Standard Arabic (MSA) completely.
7. No emojis, no offensive content, no propaganda.
8. Keep utterances concise (prefer 1–2 sentences max unless context demands otherwise).
9. Do NOT add any fields outside the schema.

SCHEMA ACCURACY RULES:
- emotion, speaker_style, speaking_rate, energy, background_noise, category MUST be selected ONLY from their respective enums.
- Do not invent new enum values.
- tags MUST be a relevant list of 2–6 short semantic labels.
- code_switching MUST be true ONLY if English words naturally appear in the text.
- contains_numbers MUST reflect whether numeric concepts are expressed (even if written in words).

BACKGROUND NOISE LOGIC:
- You must add BACKGROUND NOISE to make realistic environment .

QUALITY BAR:
- Utterances must feel like real spoken Egyptian speech in everyday scenarios.
- Avoid robotic phrasing, over-explanation, or unnatural wording.
- Prefer clarity, brevity, and realism over creativity.

SELF-CHECK (DO BEFORE FINAL OUTPUT):
- Is the JSON valid and schema-compliant?
- Are all enum values correct?
- Is the dialect strictly Egyptian Arabic?
- Are numbers written as words?
- Is there any extra text outside JSON? (must be none)

Return ONLY the final JSON object.
Take a deep breath and work on this problem step-by-step.

"""

In [288]:
structured_model =  model.with_structured_output(LLMGeneratedSchema)

# try async

In [289]:
async def generate_data(topic: str):

    start_time = datetime.utcnow()

    logger.info(f"🚀 Start generating topic: {topic}")

    messages = [
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=f"Generate one sample about topic: {topic}")
    ]

    try:
        response = await structured_model.ainvoke(messages)

        raw_data = response.model_dump()

        # -------------------------
        # Normalize
        # -------------------------
        raw_data["text"] = normalize_text(raw_data["text"])

        # -------------------------
        # Validate digits
        # -------------------------
        status = "generated"

        if not validate_no_digits(raw_data["text"]):
            status = "rejected"
            logger.warning(f"⚠️ Rejected due to digits: {topic}")

        # -------------------------
        # Schema validation
        # -------------------------
        validated = LLMGeneratedSchema(**raw_data)

        speaker_id = random.choice(["1", "2", "3", "4"])

        final_record = {
            "id": str(uuid4()),
            "schema_version": SCHEMA_VERSION,
            "created_at": datetime.utcnow().isoformat(),

            "status": status,
            "review_status": "pending",
            "source": MODEL_NAME,
            "speaker_id": speaker_id,

            **validated.model_dump()
        }

        # -------------------------
        # Save to file (IMPORTANT)
        # -------------------------
        save_to_jsonl(final_record)

        # -------------------------
        # Logging success
        # -------------------------
        duration = (datetime.utcnow() - start_time).total_seconds()

        logger.success(
            f"✅ Done topic={topic} | status={status} | time={duration:.2f}s"
        )

        return final_record

    except Exception as e:

        logger.error(f"❌ Error in topic={topic} | error={str(e)}")

        return None

In [239]:
selected_cat =  random.choice(list(Category)).value

In [240]:
selected_cat

'healthcare'

In [241]:
async def generate_batch(topics: list[str]):

    logger.info(f"🔥 Starting batch with {len(topics)} topics")

    tasks = [generate_data(t) for t in topics]

    results = await asyncio.gather(*tasks)

    # filter failed ones
    results = [r for r in results if r is not None]

    logger.info(f"🏁 Batch completed: {len(results)} successful samples")

    return results

In [242]:
all_cat

['daily_conversation',
 'delivery',
 'shopping',
 'transportation',
 'customer_support',
 'education',
 'healthcare',
 'news',
 'restaurant',
 'emergency']

In [244]:
results = await generate_batch(all_cat[1:4])

2026-05-11 02:12:38.624 | INFO     | __main__:generate_batch:3 - 🔥 Starting batch with 3 topics
C:\Users\DELL\AppData\Local\Temp\ipykernel_13116\2652860848.py:3: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  start_time = datetime.utcnow()
2026-05-11 02:12:38.625 | INFO     | __main__:generate_data:5 - 🚀 Start generating topic: delivery
2026-05-11 02:12:38.626 | INFO     | __main__:generate_data:5 - 🚀 Start generating topic: shopping
2026-05-11 02:12:38.627 | INFO     | __main__:generate_data:5 - 🚀 Start generating topic: transportation
C:\Users\DELL\AppData\Local\Temp\ipykernel_13116\2652860848.py:41: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "created_at": datetime.utcn

In [233]:
results

[{'id': 'fc478e2d-d38e-45b2-86ab-ff7d8cc54ac1',
  'schema_version': '1.0',
  'created_at': '2026-05-10T23:01:46.311517',
  'status': 'generated',
  'review_status': 'pending',
  'source': 'gemini-2.5-flash-lite',
  'speaker_id': '4',
  'text': 'ايه الاخبار يا صاحبي، عامل ايه؟',
  'emotion': <Emotion.neutral: 'speak naturally and calmly'>,
  'speaker_style': <SpeakerStyle.casual: 'use a casual conversational tone'>,
  'speaking_rate': <SpeakingRate.normal: 'speaking_rate: normal'>,
  'energy': <EnergyLevel.medium: 'with moderate energy'>,
  'background_noise': <BackgroundNoise.none: 'clean audio'>,
  'category': <Category.daily_conversation: 'daily_conversation'>,
  'code_switching': False,
  'contains_numbers': False,
  'tags': ['greetings', 'friends']},
 {'id': 'bb60a4e6-9289-4fb8-bc11-189a839cb84e',
  'schema_version': '1.0',
  'created_at': '2026-05-10T23:01:46.128292',
  'status': 'generated',
  'review_status': 'pending',
  'source': 'gemini-2.5-flash-lite',
  'speaker_id': '3',
 

# try batching

In [ ]:
all_cat = [ cat.value for cat in list(Category)]

In [ ]:
all_cat[0]

'daily_conversation'

In [ ]:
messages_1 = [
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=f"Generate one sample about topic: {all_cat[0]}")
    ]

messages_2 = [
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=f"Generate one sample about topic: {all_cat[1]}")
    ]

In [ ]:
structured_model.abatch([
    messages_1,
    messages_2
])

[LLMGeneratedSchema(text='يا جدعان عاملة ايه الدنيا معاكم؟', emotion=<Emotion.neutral: 'speak naturally and calmly'>, speaker_style=<SpeakerStyle.casual: 'use a casual conversational tone'>, speaking_rate=<SpeakingRate.normal: 'speaking_rate: normal'>, energy=<EnergyLevel.medium: 'with moderate energy'>, background_noise=<BackgroundNoise.none: 'clean audio'>, category=<Category.daily_conversation: 'daily_conversation'>, code_switching=False, contains_numbers=False, tags=['greeting', 'daily life']),
 LLMGeneratedSchema(text='يا باشا هو الطلب هيوصل امتى بالظبط؟', emotion=<Emotion.confused: 'speak with confusion and uncertainty'>, speaker_style=<SpeakerStyle.phone_call: 'speak like a natural phone conversation'>, speaking_rate=<SpeakingRate.normal: 'speaking_rate: normal'>, energy=<EnergyLevel.medium: 'with moderate energy'>, background_noise=<BackgroundNoise.none: 'clean audio'>, category=<Category.delivery: 'delivery'>, code_switching=False, contains_numbers=False, tags=['delivery time'

# synthetic data generation class

In [290]:
import asyncio
import json
import random
from pathlib import Path
from uuid import uuid4
from datetime import datetime

from loguru import logger
from langchain_core.messages import SystemMessage, HumanMessage


class SyntheticSpeechDatasetGenerator:

    def __init__(
        self,
        structured_model,
        schema_class,
        system_prompt: str,
        model_name: str,
        schema_version: str = "1.0",
        output_file: str = "dataset_output.jsonl",
        max_concurrent_tasks: int = 10,
    ):

        self.structured_model = structured_model
        self.schema_class = schema_class
        self.system_prompt = system_prompt
        self.model_name = model_name
        self.schema_version = schema_version

        self.output_file = Path(output_file)
        self.output_file.parent.mkdir(parents=True, exist_ok=True)

        self.semaphore = asyncio.Semaphore(max_concurrent_tasks)

        logger.info("✅ SyntheticSpeechDatasetGenerator initialized")

    # =====================================================
    # Save JSONL
    # =====================================================

    def save_to_jsonl(self, record: dict):

        with open(self.output_file, "a", encoding="utf-8") as f:
            f.write(
                json.dumps(
                    record,
                    ensure_ascii=False,
                    default=str
                ) + "\n"
            )

    # =====================================================
    # Process & Validate Record
    # =====================================================

    def process_record(self, raw_data: dict) -> dict:

        # -------------------------
        # Normalize text
        # -------------------------
        raw_data["text"] = normalize_text(raw_data["text"])

        # -------------------------
        # Validate digits
        # -------------------------
        status = "generated"

        if not validate_no_digits(raw_data["text"]):
            status = "rejected"

        # -------------------------
        # Schema validation
        # -------------------------
        validated = self.schema_class(**raw_data)

        speaker_id = random.choice(["1", "2", "3", "4"])

        final_record = {
            "id": str(uuid4()),
            "schema_version": self.schema_version,
            "created_at": datetime.utcnow().isoformat(),

            "status": status,
            "review_status": "pending",
            "source": self.model_name,
            "speaker_id": speaker_id,

            **validated.model_dump()
        }

        return final_record

    # =====================================================
    # Generate ONE sample
    # =====================================================

    async def generate_sample(self, topic: str):

        async with self.semaphore:

            start_time = datetime.utcnow()

            logger.info(f"🚀 Generating sample | topic={topic}")

            messages = [
                SystemMessage(content=self.system_prompt),
                HumanMessage(
                    content=f"Generate one sample about topic: {topic}"
                )
            ]

            try:

                response = await self.structured_model.ainvoke(messages)

                raw_data = response.model_dump()

                final_record = self.process_record(raw_data)

                # -------------------------
                # Save
                # -------------------------
                self.save_to_jsonl(final_record)

                duration = (
                    datetime.utcnow() - start_time
                ).total_seconds()

                logger.success(
                    f"✅ Sample generated | "
                    f"topic={topic} | "
                    f"status={final_record['status']} | "
                    f"time={duration:.2f}s"
                )

                return final_record

            except Exception as e:

                logger.error(
                    f"❌ Failed sample generation | "
                    f"topic={topic} | "
                    f"error={str(e)}"
                )

                return None

    # =====================================================
    # Generate MULTIPLE samples in parallel
    # =====================================================

    async def generate_parallel(self, topics: list[str]):

        logger.info(
            f"🔥 Starting parallel generation | "
            f"topics={len(topics)}"
        )

        tasks = [
            self.generate_sample(topic)
            for topic in topics
        ]

        results = await asyncio.gather(*tasks)

        results = [
            r for r in results
            if r is not None
        ]

        logger.success(
            f"🏁 Parallel generation completed | "
            f"successful={len(results)}"
        )

        return results

    # =====================================================
    # Generate BATCHES in parallel
    # =====================================================

    async def generate_batches(
        self,
        categories: list[str],
        samples_per_category: int = 5,
        batch_size: int = 10
    ):

        # samples_per_category = len(categories)

        logger.info(
            f"🚀 Starting generation | "
            f"categories={len(categories)} | "
            f"samples_per_category={samples_per_category} | "
            f"expected_samples={len(categories) * samples_per_category}"
        )

        all_results = []

        # =====================================================
        # EXPAND categories
        # =====================================================

        expanded_categories = []

        for _ in range(samples_per_category):

            for category in categories:

                expanded_categories.append(category)

        random.shuffle(expanded_categories)

        logger.info(
            f"📦 Expanded categories to "
            f"{len(expanded_categories)} total prompts"
        )

        # =====================================================
        # Split into provider batches
        # =====================================================

        provider_batches = [
            expanded_categories[i:i + batch_size]
            for i in range(0, len(expanded_categories), batch_size)
        ]

        logger.info(
            f"⚡ Created {len(provider_batches)} provider batches"
        )

        # =====================================================
        # Process provider batches
        # =====================================================

        for batch_index, batch_categories in enumerate(provider_batches):

            logger.info(
                f"🚀 Processing provider batch "
                f"{batch_index + 1}/{len(provider_batches)}"
            )

            try:

                batch_messages = []

                for topic in batch_categories:

                    messages = [
                        SystemMessage(content=self.system_prompt),
                        HumanMessage(
                            content=f"Generate one sample about topic: {topic}"
                        )
                    ]

                    batch_messages.append(messages)

                # ==========================================
                # PROVIDER BATCH CALL
                # ==========================================

                responses = await self.structured_model.abatch(
                    batch_messages
                )

                # ==========================================
                # Process responses
                # ==========================================

                for response in responses:

                    try:

                        raw_data = response.model_dump()

                        final_record = self.process_record(raw_data)

                        self.save_to_jsonl(final_record)

                        all_results.append(final_record)

                        logger.success(
                            f"✅ Saved sample | "
                            f"category={final_record['category']}"
                        )

                    except Exception as e:

                        logger.error(
                            f"❌ Failed processing response | "
                            f"error={str(e)}"
                        )

            except Exception as e:

                logger.error(
                    f"❌ Provider batch failed | "
                    f"batch_index={batch_index} | "
                    f"error={str(e)}"
                )

        logger.success(
            f"🏁 Generation completed | "
            f"total_samples={len(all_results)}"
        )

        return all_results

In [291]:
generator = SyntheticSpeechDatasetGenerator(
    structured_model=structured_model,
    schema_class=LLMGeneratedSchema,
    system_prompt=SYSTEM_PROMPT,
    model_name=MODEL_NAME,
    output_file="data/synthetic_dataset.jsonl",
    max_concurrent_tasks=5
)

2026-05-11 10:23:46.268 | INFO     | __main__:__init__:36 - ✅ SyntheticSpeechDatasetGenerator initialized


In [292]:
result = await generator.generate_sample(
    "delivery"
)

C:\Users\DELL\AppData\Local\Temp\ipykernel_13116\3547402861.py:102: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  start_time = datetime.utcnow()
2026-05-11 10:23:46.632 | INFO     | __main__:generate_sample:104 - 🚀 Generating sample | topic=delivery
C:\Users\DELL\AppData\Local\Temp\ipykernel_13116\3547402861.py:82: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "created_at": datetime.utcnow().isoformat(),
C:\Users\DELL\AppData\Local\Temp\ipykernel_13116\3547402861.py:127: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  d

In [293]:
result

{'id': '36542274-be3e-4f26-a243-c8c08f1b2b0a',
 'schema_version': '1.0',
 'created_at': '2026-05-11T07:23:48.150987',
 'status': 'generated',
 'review_status': 'pending',
 'source': 'gemini-2.5-flash-lite',
 'speaker_id': '3',
 'text': 'يا باشا هو الطلب وصل ولا لسه؟',
 'emotion': <Emotion.neutral: 'speak naturally and calmly'>,
 'speaker_style': <SpeakerStyle.phone_call: 'speak like a natural phone conversation'>,
 'speaking_rate': <SpeakingRate.normal: 'speaking_rate: normal'>,
 'energy': <EnergyLevel.medium: 'with moderate energy'>,
 'background_noise': <BackgroundNoise.street_noise: 'background street noise'>,
 'category': <Category.delivery: 'delivery'>,
 'code_switching': False,
 'contains_numbers': False,
 'tags': ['delivery', 'order status', 'waiting']}

In [252]:
topics = [
    "delivery",
    "banking",
    "shopping",
    "emergency"
]

results = await generator.generate_parallel(topics)

2026-05-11 09:48:53.230 | INFO     | __main__:generate_parallel:155 - 🔥 Starting parallel generation | topics=4
C:\Users\DELL\AppData\Local\Temp\ipykernel_13116\838615475.py:102: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  start_time = datetime.utcnow()
2026-05-11 09:48:53.232 | INFO     | __main__:generate_sample:104 - 🚀 Generating sample | topic=delivery
2026-05-11 09:48:53.235 | INFO     | __main__:generate_sample:104 - 🚀 Generating sample | topic=banking
2026-05-11 09:48:53.236 | INFO     | __main__:generate_sample:104 - 🚀 Generating sample | topic=shopping
2026-05-11 09:48:53.237 | INFO     | __main__:generate_sample:104 - 🚀 Generating sample | topic=emergency
C:\Users\DELL\AppData\Local\Temp\ipykernel_13116\838615475.py:82: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future 

In [254]:
len(results)

4

In [255]:
all_categories = [
    cat.value
    for cat in list(Category)
]

In [260]:
len(all_categories)

10

In [286]:


results = await generator.generate_batches(
    categories=all_categories[0:1],
    samples_per_category=2,
    batch_size=10
)

2026-05-11 10:21:12.269 | INFO     | __main__:generate_batches:192 - 🚀 Starting generation | categories=1 | samples_per_category=2 | expected_samples=2
2026-05-11 10:21:12.271 | INFO     | __main__:generate_batches:215 - 📦 Expanded categories to 2 total prompts
2026-05-11 10:21:12.271 | INFO     | __main__:generate_batches:229 - ⚡ Created 1 provider batches
2026-05-11 10:21:12.272 | INFO     | __main__:generate_batches:239 - 🚀 Processing provider batch 1/1
C:\Users\DELL\AppData\Local\Temp\ipykernel_13116\3547402861.py:82: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "created_at": datetime.utcnow().isoformat(),
2026-05-11 10:21:14.062 | SUCCESS  | __main__:generate_batches:283 - ✅ Saved sample | category=Category.daily_conversation
2026-05-11 10:21:14.064 | SUCCESS  | __main__:generate_batches:283 - ✅ Saved sample | category=Cate

In [259]:
len(results)

10

In [ ]:
1